# SchoolGPT From Scratch (Corrected Minimal Pipeline)

This notebook shows a **clean pipeline** for training a small GPT-style model from scratch:

1. Convert JSON dataset → plain text
2. Train SentencePiece tokenizer
3. Encode tokens
4. Build Dataset + DataLoader
5. Implement GPT with positional embeddings + causal mask
6. Train model
7. Generate text


In [1]:
# Install dependencies (run once)
import sys
!{sys.executable} -m pip install sentencepiece tqdm torch

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Lenovo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# Convert JSON dataset to training text
import json

with open('school_gpt_dataset.json') as f:
    data = json.load(f)

with open('train.txt','w',encoding='utf-8') as f:
    for item in data:
        text = f"""
<instruction>
{item['instruction']}

<input>
{item['input']}

<response>
{item['output']}
"""
        f.write(text.strip()+"\n\n")

print('Text dataset created')

Text dataset created


In [4]:
# Train tokenizer
import sentencepiece as spm

spm.SentencePieceTrainer.train(
    input='train.txt',
    model_prefix='school_tokenizer',
    vocab_size=8000
)

print('Tokenizer trained')

Tokenizer trained


In [5]:
# Encode tokens
import sentencepiece as spm

sp = spm.SentencePieceProcessor()
sp.load('school_tokenizer.model')

with open('train.txt','r',encoding='utf-8') as f:
    text = f.read()

tokens = sp.encode(text)

# limit tokens for faster training
tokens = tokens[:30000]

print('Total tokens:', len(tokens))

Total tokens: 30000


In [6]:
# Dataset + DataLoader
import torch
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):

    def __init__(self, tokens, seq_len=128):
        self.tokens = tokens
        self.seq_len = seq_len

    def __len__(self):
        return len(self.tokens) - self.seq_len

    def __getitem__(self, idx):
        x = self.tokens[idx:idx+self.seq_len]
        y = self.tokens[idx+1:idx+self.seq_len+1]
        return torch.tensor(x), torch.tensor(y)

dataset = TextDataset(tokens)

data_loader = DataLoader(dataset,batch_size=16,shuffle=True)

print('Steps per epoch:', len(data_loader))

Steps per epoch: 1867


In [7]:
# GPT Model with positional embeddings and causal mask
import torch.nn as nn
import torch.nn.functional as F

class GPT(nn.Module):

    def __init__(self,vocab_size,embed_size=256,seq_len=128,heads=8,layers=6):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size,embed_size)
        self.position_embedding = nn.Embedding(seq_len,embed_size)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_size,
            nhead=heads,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer,layers)

        self.fc_out = nn.Linear(embed_size,vocab_size)

        self.seq_len = seq_len

    def forward(self,x):

        B,T = x.shape

        positions = torch.arange(0,T,device=x.device).unsqueeze(0)

        x = self.token_embedding(x) + self.position_embedding(positions)

        mask = torch.triu(torch.ones(T,T,device=x.device),diagonal=1).bool()

        x = self.transformer(x,mask)

        logits = self.fc_out(x)

        return logits


In [8]:
# Training
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = GPT(vocab_size=8000).to(device)

optimizer = torch.optim.Adam(model.parameters(),lr=3e-4)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(3):

    for step,(x,y) in enumerate(tqdm(data_loader)):

        x = x.to(device)
        y = y.to(device)

        outputs = model(x)

        loss = loss_fn(
            outputs.reshape(-1,outputs.shape[-1]),
            y.reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 200 == 0:
            print('epoch',epoch,'step',step,'loss',loss.item())

torch.save(model.state_dict(),'school_gpt_weights.pth')

print('Training complete')

  0%|          | 1/1867 [00:02<1:13:24,  2.36s/it]

epoch 0 step 0 loss 9.14959716796875


 11%|█         | 201/1867 [07:47<1:13:50,  2.66s/it]

epoch 0 step 200 loss 2.8276784420013428


 21%|██▏       | 401/1867 [16:29<1:05:24,  2.68s/it]

...

epoch 2 step 1600 loss 0.04763675481081009


 96%|█████████▋| 1801/1867 [38:40<01:25,  1.29s/it]

epoch 2 step 1800 loss 0.0379251129925251


100%|██████████| 1867/1867 [40:05<00:00,  1.29s/it]

Training complete


In [28]:
def generate(prompt, max_new_tokens=50):
    model.eval()
    tokens = sp.encode(prompt)
    x = torch.tensor(tokens).unsqueeze(0).to(device)

    for _ in range(max_new_tokens):
        logits = model(x)
        probs = torch.softmax(logits[:, -1, :] / 0.8, dim=-1)  # fixed dim
        next_token = torch.multinomial(probs, 1)               # shape [1, 1]
        x = torch.cat([x, next_token], dim=1)                  # safe concat

    output = sp.decode(x[0].tolist())
    return output


print(generate("""
<instruction>
Answer the question using the context

<input>
Context: Gravity is the force that attracts objects with mass toward each other.
Question: What is gravity?

<response>
"""))

<instruction> Answer the question using the context <input> Context: Gravity is the force that attracts objects with mass toward each other. Question: What is gravity? <response> 14 <instruction> Answer the question using the context <input> Context: The university is the major seat of the Congregation of Holy Cross (albeit not its official headquarters, which are in Rome). Its main seminary,


# Finetuning our model

In [20]:
input_file = "train.txt"
output_file = "dataset_fixed.txt"

with open(input_file, encoding="utf-8") as f:
    text = f.read()

samples = text.split("Instruction:")

fixed = []

for s in samples:
    s = s.strip()
    if s:
        fixed.append("response: " + s + "\n<END>\n")

with open(output_file, "w", encoding="utf-8") as f:
    f.write("\n".join(fixed))